# FastSpeech 2 Clean-Room Master Pipeline (English & Telugu)

This notebook implements the complete, robust end-to-end FastSpeech 2 pipeline from scratch, fulfilling all requirements of **FastSpeech2_Assignment.pdf**.

### Pipeline Structure:
- **Cell 1**: Mount Google Drive, Clone/Pull Repo, Install Dependencies (`tgt`, etc.), and Extract HiFi-GAN Vocoder.
- **Cell 2**: Install Montreal Forced Aligner (MFA 2.2.17) in MicroMamba.
- **Cell 3**: English End-to-End Pipeline (~5h train / ~1h val from `ai4bharat/Svarah`, MFA alignment, pause preservation, Drive backup).
- **Cell 4**: Telugu End-to-End Pipeline (~5h train / ~1h val from `SPRINGLab/IndicTTS_Telugu`, MFA acoustic training & alignment, pause preservation, Drive backup).
- **Cell 5**: Vocoder Quality Gate — verify ground-truth mel audio reconstructions with interactive audio players.
- **Cell 6**: Fast 64-Clip Overfit Diagnostic (2,000 steps) — listen to free-running speech before full training.
- **Cell 7**: Full Independent Production Training (60,000 steps each) with real-time checkpoint & log persistence to Drive.
- **Cell 8**: Synthesize all 10 Assignment Test Sentences (E1–E4, T1–T4), generate 30 Variance Adaptor Plots, 4 Loss Curves, and Package Final Submission Zip.

## Cell 1: Environment Setup, Drive Mount, Dependencies & Vocoder Extraction

In [ ]:
import os, sys, shutil, subprocess
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/FastSpeech2_CleanRun'
os.makedirs(BASE_DIR, exist_ok=True)
for sub in ['English/checkpoints', 'English/logs', 'Telugu/checkpoints', 'Telugu/logs', 'Results/English', 'Results/Telugu', 'diagnostics']:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

# 2. Clone or pull repository
REPO_DIR = '/content/FastSpeech2-English-Telugu'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SandeshAwate6055/FastSpeech2-English-Telugu.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# 3. Install Python dependencies
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt
!pip install -q tgt
!python -c "import nltk; nltk.download('averaged_perceptron_tagger', quiet=True); nltk.download('cmudict', quiet=True)"

# 4. Extract Universal HiFi-GAN Vocoder in English and Telugu folders
for lang in ['English', 'Telugu']:
    hifi_path = f"{REPO_DIR}/{lang}/hifigan/generator_universal.pth.tar"
    if not os.path.exists(hifi_path):
        if not os.path.exists("/tmp/ming_fs2"):
            !git clone --depth 1 https://github.com/ming024/FastSpeech2.git /tmp/ming_fs2
        !unzip -q -o /tmp/ming_fs2/hifigan/generator_universal.pth.tar.zip -d {REPO_DIR}/{lang}/hifigan/
        print(f"Extracted generator_universal.pth.tar for {lang}")

print("\nSUCCESS: Cell 1 Complete! Repo, Dependencies, Drive, and Vocoder are ready.")


## Cell 2: Install Montreal Forced Aligner (MFA 2.2.17)

In [ ]:
import os

if not os.path.exists("/usr/local/bin/micromamba"):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
    !mv -f bin/micromamba /usr/local/bin/micromamba

if not os.path.exists("/root/.local/share/mamba/envs/mfa"):
    !/usr/local/bin/micromamba create -y -n mfa -c conda-forge montreal-forced-aligner=2.2.17 openfst=1.8.2 kaldi=5.5.1068 "joblib<1.4" "setuptools<81" python=3.10

!/usr/local/bin/micromamba run -n mfa mfa version
print("\nSUCCESS: Cell 2 Complete! MFA 2.2.17 is installed.")


## Cell 3: English End-to-End Pipeline (Svarah ~5h train / ~1h val, MFA Align, Preprocess & Backup)

In [ ]:
import os
from getpass import getpass

REPO_DIR = '/content/FastSpeech2-English-Telugu'
BASE_DIR = '/content/drive/MyDrive/FastSpeech2_CleanRun'
os.chdir(f"{REPO_DIR}/English")

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face Token: ")

# 1. Data selection (~5h train, ~1h val, seed 42)
!python prepare_data.py --train_hours 5.0 --val_hours 1.0 --seed 42

# 2. Montreal Forced Alignment
!/usr/local/bin/micromamba run -n mfa mfa model download acoustic english_us_arpa
!/usr/local/bin/micromamba run -n mfa mfa align \
    raw_data \
    lexicon/english_svarah_mfa.dict \
    english_us_arpa \
    preprocessed_data/TextGrid \
    --clean --single_speaker --num_jobs 4

# 3. Preprocess with pause preservation and train-only stats
!python preprocess.py --config config.yaml

# 4. Backup to Google Drive
!tar -czf {BASE_DIR}/English/English_preprocessed.tar.gz preprocessed_data lexicon raw_data/data_stats.json raw_data/train_filelist.txt raw_data/val_filelist.txt
print("\nSUCCESS: English Preprocessing Complete and Backed Up to Drive!")


## Cell 4: Telugu End-to-End Pipeline (IndicTTS ~5h train / ~1h val, MFA Train & Align, Preprocess & Backup)

In [ ]:
import os, shutil

REPO_DIR = '/content/FastSpeech2-English-Telugu'
BASE_DIR = '/content/drive/MyDrive/FastSpeech2_CleanRun'
os.chdir(f"{REPO_DIR}/Telugu")

# 1. Ensure latest code with import re
!git -C {REPO_DIR} pull

# 2. Data selection (~5h train, ~1h val)
!python prepare_data.py --train_hours 5.0 --val_hours 1.0 --seed 42

# 3. Build train-only corpus for MFA acoustic training
te_train_corpus = "/content/te_mfa_train_corpus"
if os.path.exists(te_train_corpus):
    shutil.rmtree(te_train_corpus)
os.makedirs(te_train_corpus, exist_ok=True)
for spk in ['te_female', 'te_male']:
    os.makedirs(f"{te_train_corpus}/{spk}", exist_ok=True)

with open("raw_data/train_filelist.txt", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            uid, spk = line.strip().split("|")[:2]
            for ext in [".wav", ".lab"]:
                src = f"raw_data/{spk}/{uid}{ext}"
                dst = f"{te_train_corpus}/{spk}/{uid}{ext}"
                if os.path.exists(src) and not os.path.exists(dst):
                    os.symlink(src, dst)

# Fast MFA config (monophone + triphone + lda, 20 iters)
fast_mfa_cfg = """beam: 10
retry_beam: 40
features:
  type: mfcc
  use_energy: false
  frame_shift: 10
training:
  - monophone:
      num_iterations: 20
      max_gaussians: 1000
  - triphone:
      num_iterations: 20
      num_leaves: 2000
      max_gaussians: 10000
  - lda:
      num_iterations: 20
      num_leaves: 2500
      max_gaussians: 15000
"""
with open("/content/fast_mfa_train.yaml", "w") as f:
    f.write(fast_mfa_cfg)

# 4. Train MFA acoustic model on train partition and align
!/usr/local/bin/micromamba run -n mfa mfa train \
    {te_train_corpus} \
    lexicon/telugu-lexicon.txt \
    te_acoustic.zip \
    --config_path /content/fast_mfa_train.yaml \
    --single_speaker --clean --num_jobs 4

!/usr/local/bin/micromamba run -n mfa mfa align \
    raw_data \
    lexicon/telugu-lexicon.txt \
    te_acoustic.zip \
    preprocessed_data/TextGrid \
    --single_speaker --clean --num_jobs 4

# 5. Feature extraction with pause preservation and train-only bounds
!python preprocess.py --config config.yaml

# 6. Backup to Google Drive
!tar -czf {BASE_DIR}/Telugu/Telugu_preprocessed.tar.gz preprocessed_data lexicon raw_data/data_stats.json raw_data/train_filelist.txt raw_data/val_filelist.txt
print("\nSUCCESS: Telugu Preprocessing Complete and Backed Up to Drive!")


## Cell 5: Vocoder Quality Gate (Audio Verification)
- Reconstructs raw waveforms from preprocessed ground-truth mel-spectrograms using Universal HiFi-GAN.
- **Listen to both outputs**: verify that the vocoder sounds natural and clean before training.

In [ ]:
import sys, os, yaml, torch, numpy as np
from IPython.display import Audio, display

REPO_DIR = '/content/FastSpeech2-English-Telugu'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Active Device:", device)

for lang in ["English", "Telugu"]:
    lang_dir = f"{REPO_DIR}/{lang}"
    sys.path.insert(0, lang_dir)
    from utils.model import get_vocoder, vocoder_infer
    cfg = yaml.safe_load(open(f"{lang_dir}/config.yaml"))
    vocoder = get_vocoder(cfg, device)
    
    train_line = open(f"{lang_dir}/preprocessed_data/train.txt").readline().strip().split("|")
    uid, spk = train_line[0], train_line[1]
    mel = np.load(f"{lang_dir}/preprocessed_data/mel/{spk}-mel-{uid}.npy")
    mel_t = torch.from_numpy(mel).float().unsqueeze(0).transpose(1, 2).to(device)
    wav = vocoder_infer(mel_t, vocoder, cfg, cfg)[0]
    
    print(f"\n=== {lang} Reconstructed Ground-Truth Audio: {uid} ({spk}) ===")
    display(Audio(wav, rate=22050))
    sys.path.pop(0)


## Cell 6: Fast 64-Clip Overfit Diagnostic (2,000 steps)
- Verifies that the FastSpeech 2 model learns duration, pitch, and energy predictors properly.
- Reconstructs free-running speech on seen training sentences in ~2 minutes.

In [ ]:
import os, sys
REPO_DIR = '/content/FastSpeech2-English-Telugu'

for lang in ["English", "Telugu"]:
    lang_dir = f"{REPO_DIR}/{lang}"
    diag_dir = f"/content/overfit_diag/{lang}"
    os.makedirs(diag_dir, exist_ok=True)
    
    os.makedirs(f"{diag_dir}/preprocessed_data", exist_ok=True)
    for sub in ["mel", "pitch", "energy", "duration", "stats.json", "speakers.json"]:
        src = f"{lang_dir}/preprocessed_data/{sub}"
        dst = f"{diag_dir}/preprocessed_data/{sub}"
        if not os.path.exists(dst):
            os.symlink(src, dst)
        
    lines = open(f"{lang_dir}/preprocessed_data/train.txt").readlines()[:64]
    with open(f"{diag_dir}/preprocessed_data/train.txt", "w") as f: f.writelines(lines)
    with open(f"{diag_dir}/preprocessed_data/val.txt", "w") as f: f.writelines(lines[:16])
    
    print(f"\n--- Running 2000-Step Diagnostic Overfit for {lang} ---")
    !python {lang_dir}/train.py --config {lang_dir}/config.yaml \
        --restore_step 0 --total_step 2000 --val_step 2000 --save_step 2000 \
        --preprocessed_path {diag_dir}/preprocessed_data \
        --ckpt_path {diag_dir}/checkpoints \
        --log_path {diag_dir}/logs \
        --result_path {diag_dir}/results


## Cell 7: Full Independent Production Training (60,000 steps each)
- Checkpoints, training loss logs, and validation loss logs are saved in real-time to Google Drive under `FastSpeech2_CleanRun/{English,Telugu}/`.

In [ ]:
import os
REPO_DIR = '/content/FastSpeech2-English-Telugu'
BASE_DIR = '/content/drive/MyDrive/FastSpeech2_CleanRun'

# English Training
print("\n================ Starting English FastSpeech 2 Full Training ================")
os.chdir(f"{REPO_DIR}/English")
!python train.py --config config.yaml --restore_step 0 --total_step 60000 \
    --val_step 1000 --save_step 5000 \
    --ckpt_path {BASE_DIR}/English/checkpoints \
    --log_path {BASE_DIR}/English/logs \
    --result_path {BASE_DIR}/Results/English

# Telugu Training
print("\n================ Starting Telugu FastSpeech 2 Full Training ================")
os.chdir(f"{REPO_DIR}/Telugu")
!python train.py --config config.yaml --restore_step 0 --total_step 60000 \
    --val_step 1000 --save_step 5000 \
    --ckpt_path {BASE_DIR}/Telugu/checkpoints \
    --log_path {BASE_DIR}/Telugu/logs \
    --result_path {BASE_DIR}/Results/Telugu


## Cell 8: Mandatory Test Sentence Synthesis, Variance Adaptor Plots & Deliverable Packaging
- Synthesizes speech for all 10 assignment sentences (E1–E4, T1–T4).
- Generates duration, pitch, and energy plots for each sentence.
- Copies loss curves (`training_loss.png`, `validation_loss.png`) and bundles everything into `Results_Deliverables_Final.zip` on Google Drive.

In [ ]:
import os, shutil
REPO_DIR = '/content/FastSpeech2-English-Telugu'
BASE_DIR = '/content/drive/MyDrive/FastSpeech2_CleanRun'

os.chdir(f"{REPO_DIR}/English")
!python inference.py --config config.yaml --speaker_id -1 \
    --ckpt_file {BASE_DIR}/English/checkpoints/best_val.pth.tar \
    --result_path {BASE_DIR}/Results/English

os.chdir(f"{REPO_DIR}/Telugu")
!python inference.py --config config.yaml --speaker_id 0 \
    --ckpt_file {BASE_DIR}/Telugu/checkpoints/best_val.pth.tar \
    --result_path {BASE_DIR}/Results/Telugu

# Sync local repo Results for GitHub tracking
for lang in ['English', 'Telugu']:
    src_res = f"{BASE_DIR}/Results/{lang}"
    dst_res = f"{REPO_DIR}/Results/{lang}"
    if os.path.exists(dst_res):
        shutil.rmtree(dst_res)
    shutil.copytree(src_res, dst_res)

os.chdir(BASE_DIR)
!zip -r Results_Deliverables_Final.zip Results/
print("\nSUCCESS: All deliverables packaged to:", f"{BASE_DIR}/Results_Deliverables_Final.zip")
